# SAM Auto-Segmentation  - pure-SAM method (stage 1 of 2)   ·  Colab / GPU

**Method: SAM (no geometric prior).** First stage of the **pure-SAM** branch of the
three-method comparison. Where the *geometric* method seeds rooms with a deterministic
watershed and *geometric+SAM* refines that watershed with prompted SAM, this method runs
**SAM in automatic "segment everything" mode** directly on the Stage-1 rasters  - **no
watershed prior, no prompts**  - and turns the resulting masks into room labels
(`scan2bim.segment_rooms_sam_auto`). This reproduces the paper's room pipeline (Albadri et
al., ISPRS Archives XLVIII-G-2025-131, §3.1).

> **Three-method comparison.** geometric (watershed only) · **SAM (this branch)** ·
> geometric+SAM (watershed prior + prompted-SAM refinement). All three write a
> `room_labels.npy` on the **same Stage-1 grid**, so `evaluation/pq_eval.ipynb` scores them
> against the same ground truth with zero resampling.

**GPU stage.** SAM's automatic mask generator needs a CUDA GPU to be practical  - run this in
**Google Colab (T4)**. Pure-SAM is meaningless without SAM, so with no backend this notebook
**fails loudly** at the segmentation step (it never fabricates rooms).

## 1  - Clone the code + mount Google Drive (Option B: everything on Colab)
The cell below clones `scan2bim` onto the runtime (or imports it if already present), then
**mounts Google Drive** and points every stage output at `MyDrive/onestruction/out/`. That
makes outputs **persist across runtime restarts** and removes the Colab→local file shuffle  -
notebook 2 reads them straight back from Drive.

> Expected Drive layout (edit `DRIVE_ROOT` if you named it differently):
> ```
> MyDrive/onestruction/
> ├── data/area1.xyz   ← the cloud (used by notebook 2)
> └── out/             ← stage_sam_auto, stage_sam_walls land here
> ```
> This stage (segmentation) needs only the 247 KB Stage-1 rasters from the repo  - **not** the
> point cloud  - so it's quick. The first run will pop a Drive auth prompt.

In [ ]:
# ===== scan2bim setup  - clone/import the code, then Drive-back the outputs (Option B) =====
import sys, os, shutil
import numpy as np

REPO_URL   = 'https://github.com/PrinceofJ/ONESTRUCTION-Point-Cloud-to-BIM.git'
BRANCH     = 'sam-auto-method'                       # clone fallback; must be pushed to origin
CLONE      = '/content/onestruction'
DRIVE_ROOT = '/content/drive/MyDrive/onestruction'   # <-- your Drive folder (data/ + out/)

try:
    import google.colab            # noqa: F401  (only importable on Colab)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    
PROJECT_ROOT = os.path.abspath('../../..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.path = [p for p in sys.path if not p.endswith('/src') and not p.endswith('\\src')]
for mod in list(sys.modules):
    if mod == 'scan2bim' or mod.startswith('scan2bim.'):
        del sys.modules[mod]

def _try_import_scan2bim():
    try:
        import scan2bim
        return scan2bim
    except ModuleNotFoundError:
        return None

scan2bim = _try_import_scan2bim()
if scan2bim is not None:                              # already on the runtime (synced / local)
    PROJECT_DIR = scan2bim.project_root(); SOURCE = 'already on runtime'
elif IN_COLAB:                                        # bare Colab box -> fetch from GitHub
    if not os.path.isdir(CLONE):
        get_ipython().system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {CLONE}')
    if not os.path.isdir(os.path.join(CLONE, 'scan2bim')):
        raise RuntimeError(f"Clone failed: push branch '{BRANCH}' first  (git push -u origin {BRANCH}).")
    sys.path.insert(0, CLONE); os.chdir(CLONE)
    import scan2bim; PROJECT_DIR = CLONE; SOURCE = f'git clone ({BRANCH})'
else:
    raise ModuleNotFoundError("scan2bim not importable locally  - run `pip install -e .` first.")

from scan2bim import artifacts as A

# ---- Drive-backed, persistent stage outputs ----
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = os.path.join(DRIVE_ROOT, 'out')        # stage outputs persist here on Drive
    os.makedirs(OUT_ROOT, exist_ok=True)
    if not (os.path.isfile(A.stage_zip(OUT_ROOT, A.STAGE1)) or
            os.path.isdir(A.stage_dir(OUT_ROOT, A.STAGE1))):     # seed Stage-1 zip from the clone
        shutil.copy(A.stage_zip(os.path.join(PROJECT_DIR, 'scan2bim_out'), A.STAGE1),
                    A.stage_zip(OUT_ROOT, A.STAGE1))
        print('seeded stage1 zip into Drive out/')
else:                                                 # local run: keep outputs in the repo
    OUT_ROOT = os.path.join(PROJECT_DIR, 'scan2bim_out')

print('scan2bim', scan2bim.__version__, '| in_colab:', IN_COLAB, '| source:', SOURCE)
print('OUT_ROOT   :', OUT_ROOT, '| stage1 present:',
      os.path.isfile(A.stage_zip(OUT_ROOT, A.STAGE1)) or os.path.isdir(A.stage_dir(OUT_ROOT, A.STAGE1)))

scan2bim 1.0.0 | in_colab: False | source: already on runtime
OUT_ROOT   : /Users/jacksonmatsumura/Documents/ONESTRUCTION-Point-Cloud-to-BIM/scan2bim_out | stage1 present: True


## 2  - Check the GPU, then install SAM 2 + the SAM 2.1 checkpoint
SAM 2's automatic mask generator needs a CUDA GPU. In Colab: **Runtime > Change runtime
type > GPU**. Without a working SAM backend the segmentation step raises a clear error
(pure-SAM does not pass through like the refinement stage).

In [ ]:
# ------------------------------- GPU check -------------------------------
try:
    import torch
    if torch.cuda.is_available():
        print('CUDA OK ->', torch.cuda.get_device_name(0))
    else:
        print('WARNING: no CUDA GPU. In Colab: Runtime > Change runtime type > GPU. '
              'Without a SAM backend the segmentation step will raise.')
except Exception as e:
    print('torch not installed yet:', e, '- install it in the next cell (Colab).')

torch not installed yet: No module named 'torch' - install it in the next cell (Colab).


In [ ]:
# ===== Install SAM 2 + download the SAM 2.1 checkpoint (Colab; runs itself, idempotent) =====
# Verified against github.com/facebookresearch/sam2. Needs python>=3.10, torch>=2.5.1,
# torchvision>=0.20.1 (preinstalled on Colab GPU runtimes). Installs once if missing and
# downloads the checkpoint once if absent. SAME backend/checkpoint as the refinement stage.
SAM_CKPT = '/content/sam2.1_hiera_large.pt'
SAM_CFG  = 'configs/sam2.1/sam2.1_hiera_l.yaml'
CKPT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'

if IN_COLAB:
    import importlib.util
    if importlib.util.find_spec('sam2') is None:
        get_ipython().system('pip install -q "git+https://github.com/facebookresearch/sam2.git"')
    if not os.path.exists(SAM_CKPT):
        get_ipython().system('wget -q -O {SAM_CKPT} {CKPT_URL}')
    print('SAM 2 ready | checkpoint:', SAM_CKPT, '| exists:', os.path.exists(SAM_CKPT))
else:
    print('Local run - skipping SAM install; the segmentation step will raise without a backend.')
print('SAM config name:', SAM_CFG)

Local run - skipping SAM install; the segmentation step will raise without a backend.
SAM config name: configs/sam2.1/sam2.1_hiera_l.yaml


## 3  - Load CFG (validated against Stage 1) + the Stage-1 rasters
`CFG` comes from the unified `load_config()` (params.yaml) and is **validated** against the
`config.json` saved inside the Stage-1 ZIP, so the SAM labels land on the exact grid the
other two methods use. Only the runtime / SAM fields (paths + backend) are overridden. The
paper's Table-1 parameters live in `params.yaml` / `Config` (`sam_points_per_side`, etc.).

In [ ]:
from PIL import Image
s1 = A.load_stage_dir(OUT_ROOT, A.STAGE1)               # shared Stage-1 occupancy rasters

CFG = scan2bim.load_config(start=PROJECT_DIR, out_root=OUT_ROOT)

# Validate we're on the same cloud + grid as Stage 1. Cross-platform guard: the stage1
# config.json may have been written on Windows ('c:\...\area1.xyz') and is being validated here
# on Linux/Colab, where os.path.basename can't split a '\\' path -> a spurious file_path
# mismatch. Pre-equalise file_path ONLY when the basenames truly match; every other geometry
# field (pixel_m, slab, voxel, up_axis, ...) is still checked strictly. (The library has the
# same fix now; this keeps the notebook working even against an older cloned package.)
_up = A.load_stage_config(s1)
def _basename(p): return str(p).replace('\\', '/').rsplit('/', 1)[-1]
if _basename(_up.get('file_path')) == _basename(CFG.file_path):
    _up = {**_up, 'file_path': CFG.file_path}
scan2bim.assert_upstream_config(CFG, _up)

# runtime / SAM overrides only (do NOT touch geometry params)
CFG.sam_backend   = 'sam2'                              # 'sam2' | 'sam3' | 'sam1'
CFG.sam_ckpt      = SAM_CKPT
CFG.sam_model_cfg = SAM_CFG
# PAPER-FAITHFUL SAM input (Albadri et al. §3.1). SAM must segment Ms  - the binary top-down of
# the cropped SLAB section ONLY (walls black, rooms empty). 'occupancy' replicates that Ms into
# 3 channels; the config default 'stack' would instead fold in the slab wall mask + coverage
# channels (coverage fills scanned interiors), making SAM mask the whole building. Scoped HERE, not in
# params.yaml, on purpose: the geometric+SAM *prompted* refinement (Notebook 4) shares
# CFG.sam_image_mode and may legitimately prefer 'stack'. Flip back to 'stack' here to ablate.
CFG.sam_image_mode = 'occupancy'
SHOW_DEBUG = True

occ      = np.array(Image.open(os.path.join(s1, A.OCC_PNG)))
wall_mask = A.load_npy(os.path.join(s1, A.WALLMASK_NPY)).astype(bool)
coverage = A.load_npy(os.path.join(s1, A.COVERAGE_NPY)).astype(bool)
tf       = A.load_transform(os.path.join(s1, A.TRANSFORM_JSON))
print('raster (H x W):', wall_mask.shape, '| pixel_m =', CFG.pixel_m,
      '| points_per_side =', CFG.sam_points_per_side,
      '| A =', CFG.sam_auto_min_room_area_m2, 'm^2', '| sam_image_mode =', CFG.sam_image_mode)

## 4  - Build the SAM image and run automatic "segment everything"
`build_sam_image(..., mode='occupancy')` builds SAM's input from **Ms only**  - the binary
top-down of the cropped slab section (walls black, room interiors empty), replicated to 3
channels. This is the paper-faithful input (§3.1); the slab `wall_mask` + `coverage`
channels are deliberately **not** fed to SAM (the coverage channel fills room interiors and
makes SAM mask the whole building). `segment_rooms_sam_auto` then runs SAM's automatic mask generator, turns
masks into rooms (room/not-room by area `A`, deterministic overlap resolution, walls
re-imposed as `-1` on the **slab's own** occupied pixels), and  - if enabled in `params.yaml`
 - does corridor reprocessing (`sam_reprocess_residual`) and/or the boundary buffer
(`sam_auto_buffer_rooms`). **Pure-SAM needs a real backend; if none built, we stop here
rather than emit an empty map.**

In [ ]:
image = scan2bim.build_sam_image(occ, wall_mask, coverage, mode=CFG.sam_image_mode)

# PAPER-FAITHFUL wall scaffold: the OCCUPIED pixels of the slab projection Ms itself
# (occupancy.png is 255 free / 0 wall); this equals the saved `wall_mask`. This keeps BOTH
# SAM's input image AND its wall scaffold derived from Ms only (§3.1) — the walls are exactly
# the black pixels SAM segments around. (coverage ~ the full-cloud footprint Mf is still used,
# but only to drop the exterior background mask SAM always emits  - retrieval/masking, not input.)
occ2d = occ[..., 0] if occ.ndim == 3 else occ
slab_walls = occ2d < 128
labels, dbg = scan2bim.segment_rooms_sam_auto(image, walls=slab_walls, coverage=coverage, cfg=CFG)
print(dbg)
if not dbg['ran']:
    raise RuntimeError(
        'Pure-SAM produced no segmentation: ' + dbg.get('reason', 'no SAM backend') +
        '. This method REQUIRES a SAM backend (run on a Colab GPU with the checkpoint '
        'installed above); it does not fall back like the refinement stage.')
print('backend', dbg['backend'], '| masks', dbg['n_masks'], '-> kept', dbg['n_kept'],
      '(void-dropped', dbg['n_void_dropped'], ') | rooms pass1', dbg['n_rooms_pass1'],
      '-> out', dbg['n_rooms_out'], '| reprocess +', dbg['reprocess']['n_added'])

## Optional  - QA plot (SAM input image vs the resulting rooms)

In [ ]:
if SHOW_DEBUG:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(image); ax[0].set_title('SAM input = Ms (slab occupancy, paper-faithful)')
    rgb = np.ones(labels.shape + (3,))
    rgb[labels == -1] = (0, 0, 0); rgb[labels == 0] = (0.93, 0.93, 0.93)
    cmap = plt.get_cmap('tab20')
    rooms = [int(x) for x in np.unique(labels) if x >= 1]
    for k, r in enumerate(rooms):
        rgb[labels == r] = cmap(k % 20)[:3]
    ax[1].imshow(rgb); ax[1].set_title('SAM rooms: %d' % len(rooms))
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

## 5  - Save room_labels on the Stage-1 grid + package the stage
Writes to `stage_sam_auto/` (distinct from the other methods, so nothing clobbers). Emitting
`room_labels.npy` on the Stage-1 grid lets `notebook_2_wall_assignment.ipynb` and
`evaluation/pq_eval.ipynb` consume it by name, exactly like the watershed output.

In [ ]:
out_dir = A.ensure_dir(A.stage_dir(OUT_ROOT, A.STAGE_SAM_AUTO))
A.save_npy(os.path.join(out_dir, A.ROOM_LABELS_NPY), labels.astype('int32'))
A.save_label_png(os.path.join(out_dir, A.ROOM_LABELS_PNG), labels)
A.save_transform(os.path.join(out_dir, A.TRANSFORM_JSON), tf)
A.save_config(os.path.join(out_dir, A.CONFIG_JSON), CFG)
zip_path = A.package_stage(OUT_ROOT, A.STAGE_SAM_AUTO)
print('packaged ->', zip_path)

## Output is already on your Drive (no download step)
Because `OUT_ROOT` lives on Drive, `stage_sam_auto` was written straight to
`MyDrive/onestruction/out/`  - it persists across runtime restarts and notebook 2 reads it
from there. (The old `files.download` browser trick doesn't work through the VSCode-Colab
connection, which is why we route through Drive instead.)

In [ ]:
# Drive-backed OUT_ROOT -> nothing to download; confirm where it landed.
print('stage_sam_auto is on your Drive at:')
print('   ', A.stage_zip(OUT_ROOT, A.STAGE_SAM_AUTO))
print('   ', A.stage_dir(OUT_ROOT, A.STAGE_SAM_AUTO), '(room_labels.npy, color PNG)')
print('Next: run notebook_2_wall_assignment.ipynb (also Colab + Drive) to get the 3-D walls.')